<a href="https://colab.research.google.com/github/Argo-Navis-Lab/scalelab/blob/main/notebooks/colab_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ScaleLab on Colab: run one experiment

This notebook only installs ScaleLab and calls `scalelab run`, the same command you run on your laptop. Everything a run writes (checkpoints, artifacts and a copy of the registry) goes to Google Drive, so it outlives the VM.

1. **Runtime → Change runtime type → T4 GPU.**
2. Fill in the settings below, then **Runtime → Run all**.
3. If the session ends before the run does, open this notebook again and **Run all** with the same settings: the run resumes from its last checkpoint (saved every 15 minutes).

> **Setting Cell**

In [1]:
import os

# The code to run: a full commit SHA (`git log -1 --format=%H`) pins it exactly.
COMMIT = "f028406cd148e7d01ee6b4365e7856ccf198dee4"  # @param {type:"string"}
EXPERIMENT = "configs/experiments/hello.yaml"  # @param {type:"string"}
OVERRIDES = ""  # @param {type:"string"}

os.environ["COMMIT"] = COMMIT
os.environ["EXPERIMENT"] = EXPERIMENT
os.environ["OVERRIDES"] = OVERRIDES
os.environ["SCALELAB_HOME"] = "/content/drive/MyDrive/scalelab"  # on Drive: outlives the VM

> **Drive Cell**

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


> **Install Cell**

In [3]:
%%bash
# Install ScaleLab at COMMIT, with exactly the locked dependencies (a few minutes: PyTorch).
set -euo pipefail
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
pip install --quiet uv
rm -rf /content/scalelab
git clone --quiet https://github.com/Argo-Navis-Lab/scalelab.git /content/scalelab
cd /content/scalelab
git checkout --quiet --detach "$COMMIT"
git log -1 --format="code: %H %s"
uv sync --locked --quiet

Tesla T4, 15360 MiB
code: f028406cd148e7d01ee6b4365e7856ccf198dee4 m3s24: fp16, measured (cost measure --change fp16)


# M3S24's cell:

> Cell M:

In [4]:
%%bash
# m3s24: fp16, measured, on a T4 runtime (the A100 has no fp16 fit). It needs cell L's fp16
# bench and refit on Drive. For each experiment, `cost measure --change fp16` first predicts its
# step in fp32 and in fp16 (as cell L's compare did) and saves that report, then runs its first
# 23 steps twice on the T4, in fp32 and then under autocast fp16 with a GradScaler, and sets the
# last 20 steps' mean laps beside the prediction: M2's exit RL (× at bands 3 and 4, from M1's
# warm starts) for m1-m3, warmstart.yaml for m1-m4. The first fp16 steps on a GPU. About 12
# minutes. Reports go to $SCALELAB_HOME/reports/ on Drive, nothing to the registry.
set -f  # the * and [3,4] below are values, not files to match
cd /content/scalelab
if ! nvidia-smi --query-gpu=name --format=csv,noheader | grep -q T4; then
  echo "cell M needs a T4 runtime"; exit 1
fi
for pair in m1:c73d1d0f4e234964 m2:71cd3add8c11e667 m3:52606c3253c9e849; do
  uv run scalelab cost measure configs/experiments/rl.yaml model=ladder_s/${pair%%:*} \
    rl.from_run=${pair#*:}-s0-2aeaf16b task.ops=* task.difficulties=[3,4] \
    --hardware t4 --change fp16 | grep -E '^(predicted|# |\| [glst][a-z]+ |- )'
done
for model in m1 m2 m3 m4; do
  uv run scalelab cost measure configs/experiments/warmstart.yaml model=ladder_s/$model \
    --hardware t4 --change fp16 | grep -E '^(predicted|# |\| [glst][a-z]+ |- )'
done

predicted:     step 1.9135 -> 1.8226 s (-4.8%), /content/drive/MyDrive/scalelab/reports/fp16-066d0df6944753c1-t4.md
# Measured: fp16, rl on t4
| generate | 1.3893 | 1.5220 | +9.6% | 1.5229 | 1.7729 | +16.4% |
| logprobs | 0.1840 | 0.0912 | -50.4% | 0.1667 | 0.0988 | -40.7% |
| train | 0.3402 | 0.2093 | -38.5% | 0.3634 | 0.2461 | -32.3% |
| step | 1.9135 | 1.8226 | -4.8% | 2.0830 | 2.1400 | +2.7% |
- speedup: predicted 1.05x, measured 0.97x, so the prediction is off by +7.9%
- verdict: the prediction missed: beyond the 4.1% error band
predicted:     step 4.3954 -> 3.2129 s (-26.9%), /content/drive/MyDrive/scalelab/reports/fp16-771bf34af837291e-t4.md
# Measured: fp16, rl on t4
| generate | 2.4894 | 2.4112 | -3.1% | 2.1784 | 2.3654 | +8.6% |
| logprobs | 0.7276 | 0.2762 | -62.0% | 0.7192 | 0.3123 | -56.6% |
| train | 1.1784 | 0.5256 | -55.4% | 1.2316 | 0.6244 | -49.3% |
| step | 4.3954 | 3.2129 | -26.9% | 4.1504 | 3.3224 | -19.9% |
- speedup: predicted 1.37x, measured 1.25x, so the predic

# MSS20's cell:

Cell L:

In [ ]:
%%bash
# m3s20: DecisionReport 2, fp16, on a T4 runtime. `bench --precision fp16` times the gemm,
# attention and decode-step suites in float16 under autocast (m3s12's fp32 bench took 29 s),
# saved beside the fp32 results in $SCALELAB_HOME/bench/t4/ on Drive. `cost calibrate` refits
# every group with them (the *.fp16 groups join). `cost validate` rechecks the fp32 runs' whole
# steps with the refit (m3s12: 4.0%) and saves the error band. `cost compare --change fp16`
# then predicts each experiment's step in fp32 and in fp16: M2's exit RL (× at bands 3 and 4)
# for m1-m3, warmstart.yaml for m1-m4. Only the bench runs on the GPU; reports go to
# $SCALELAB_HOME/reports/, nothing to the registry. The greps keep each report's table rows
# and bullets.
set -f  # the * and [3,4] below are values, not files to match
cd /content/scalelab
nvidia-smi --query-gpu=name --format=csv,noheader | grep -q T4 || { echo "needs a T4"; exit 1; }
uv run scalelab bench --hardware t4 --precision fp16
uv run scalelab cost calibrate --hardware t4
uv run scalelab cost validate --hardware t4 | grep -E '^(step |verdict|saved)'
for model in m1 m2 m3; do
  uv run scalelab cost compare configs/experiments/rl.yaml model=ladder_s/$model \
    task.ops=* task.difficulties=[3,4] --change fp16 --hardware t4 \
    | grep -E '^(# |\| [glst][a-z]+ |- )'
done
for model in m1 m2 m3 m4; do
  uv run scalelab cost compare configs/experiments/warmstart.yaml model=ladder_s/$model \
    --change fp16 --hardware t4 | grep -E '^(# |\| [glst][a-z]+ |- )'
done

hardware:      t4
models:        ladder_s/m1 ladder_s/m2 ladder_s/m3 ladder_s/m4
precision:     fp16
results:       /content/drive/MyDrive/scalelab/bench/t4
gemm:          107 cases: 107 timings in 2 s
attention:     112 cases: 160 timings in 5 s
decode:        64 cases: 64 timings in 22 s
hardware:      t4
measured on:   torch 2.14.1, cuda 13.0, device Tesla T4
samples:       754 from /content/drive/MyDrive/scalelab/bench/t4
group           compute  memory     launch   (1.000: never the limit)
gemm              0.508   0.348   19.02 us
gemv-decode       1.000   0.926  113.85 us
attention         0.106   0.322   61.11 us
gemm.fp16         0.367   0.550   16.05 us
gemv-decode.fp16   1.000   1.000  148.60 us
attention.fp16    1.000   0.492   66.07 us
copy              1.000   0.761    6.61 us
norm              1.000   0.232   51.59 us
norm.grad         1.000   0.085  337.34 us
rope              1.000   0.155   66.72 us
rope.grad         1.000   0.293  274.32 us
act               1.000   

# M3S17's cell:

## On T4:

> Cell K:

In [ ]:
%%bash
# m3s17: the padding cut, measured. Run it once on a T4 runtime and once on an A100 one. For
# each experiment, `cost measure` first predicts its step without and with the cut (as cell J's
# compare did) and saves that report, then runs its first 23 steps twice on this GPU, without
# the cut and with it, and sets the last 20 steps' mean laps beside the prediction: M2's exit RL
# (× at bands 3 and 4, from M1's warm starts) for m1-m3, warmstart.yaml for m1-m4. About 15
# minutes on the T4. Reports go to $SCALELAB_HOME/reports/ on Drive, nothing to the registry.
# The greps keep each report's prediction line, table rows and bullets.
set -f  # the * and [3,4] below are values, not files to match
cd /content/scalelab
hardware=t4
if nvidia-smi --query-gpu=name --format=csv,noheader | grep -q A100; then hardware=a100-40g; fi
for pair in m1:c73d1d0f4e234964 m2:71cd3add8c11e667 m3:52606c3253c9e849; do
  uv run scalelab cost measure configs/experiments/rl.yaml model=ladder_s/${pair%%:*} \
    rl.from_run=${pair#*:}-s0-2aeaf16b task.ops=* task.difficulties=[3,4] \
    --hardware $hardware | grep -E '^(predicted|# |\| [glst][a-z]+ |- )'
done
for model in m1 m2 m3 m4; do
  uv run scalelab cost measure configs/experiments/warmstart.yaml model=ladder_s/$model \
    --hardware $hardware | grep -E '^(predicted|# |\| [glst][a-z]+ |- )'
done

predicted:     step 1.9131 -> 1.8543 s (-3.1%), /content/drive/MyDrive/scalelab/reports/padding-cut-066d0df6944753c1-t4.md
# Measured: padding-cut, rl on t4
| generate | 1.4018 | 1.4018 | +0.0% | 1.5388 | 1.5302 | -0.6% |
| logprobs | 0.1790 | 0.1697 | -5.2% | 0.1692 | 0.1571 | -7.2% |
| train | 0.3323 | 0.2828 | -14.9% | 0.3674 | 0.3023 | -17.7% |
| step | 1.9131 | 1.8543 | -3.1% | 2.0966 | 2.0115 | -4.1% |
- speedup: predicted 1.03x, measured 1.04x, so the prediction is off by -1.0%
- verdict: the prediction held: within the 4.0% error band
predicted:     step 4.3987 -> 4.1048 s (-6.7%), /content/drive/MyDrive/scalelab/reports/padding-cut-771bf34af837291e-t4.md
# Measured: padding-cut, rl on t4
| generate | 2.5313 | 2.5313 | +0.0% | 2.2674 | 2.2297 | -1.7% |
| logprobs | 0.7125 | 0.6133 | -13.9% | 0.7441 | 0.6459 | -13.2% |
| train | 1.1549 | 0.9602 | -16.9% | 1.2883 | 1.0846 | -15.8% |
| step | 4.3987 | 4.1048 | -6.7% | 4.3206 | 3.9824 | -7.8% |
- speedup: predicted 1.07x, measured 

## On A100:

> Cell K:

In [ ]:
%%bash
# m3s17: the padding cut, measured. Run it once on a T4 runtime and once on an A100 one. For
# each experiment, `cost measure` first predicts its step without and with the cut (as cell J's
# compare did) and saves that report, then runs its first 23 steps twice on this GPU, without
# the cut and with it, and sets the last 20 steps' mean laps beside the prediction: M2's exit RL
# (× at bands 3 and 4, from M1's warm starts) for m1-m3, warmstart.yaml for m1-m4. About 15
# minutes on the T4. Reports go to $SCALELAB_HOME/reports/ on Drive, nothing to the registry.
# The greps keep each report's prediction line, table rows and bullets.
set -f  # the * and [3,4] below are values, not files to match
cd /content/scalelab
hardware=t4
if nvidia-smi --query-gpu=name --format=csv,noheader | grep -q A100; then hardware=a100-40g; fi
for pair in m1:c73d1d0f4e234964 m2:71cd3add8c11e667 m3:52606c3253c9e849; do
  uv run scalelab cost measure configs/experiments/rl.yaml model=ladder_s/${pair%%:*} \
    rl.from_run=${pair#*:}-s0-2aeaf16b task.ops=* task.difficulties=[3,4] \
    --hardware $hardware | grep -E '^(predicted|# |\| [glst][a-z]+ |- )'
done
for model in m1 m2 m3 m4; do
  uv run scalelab cost measure configs/experiments/warmstart.yaml model=ladder_s/$model \
    --hardware $hardware | grep -E '^(predicted|# |\| [glst][a-z]+ |- )'
done

predicted:     step 1.3968 -> 1.4193 s (+1.6%), /content/drive/MyDrive/scalelab/reports/padding-cut-066d0df6944753c1-a100-40g.md
# Measured: padding-cut, rl on a100-40g
| generate | 1.2556 | 1.2556 | +0.0% | 1.4828 | 1.4778 | -0.3% |
| logprobs | 0.0336 | 0.0639 | +90.0% | 0.0397 | 0.0845 | +113.0% |
| train | 0.1076 | 0.0998 | -7.2% | 0.1766 | 0.1680 | -4.9% |
| step | 1.3968 | 1.4193 | +1.6% | 1.7205 | 1.7518 | +1.8% |
- speedup: predicted 0.98x, measured 0.98x, so the prediction is off by +0.2%
- verdict: the prediction held: within the 13.2% error band
predicted:     step 2.3830 -> 2.3753 s (-0.3%), /content/drive/MyDrive/scalelab/reports/padding-cut-771bf34af837291e-a100-40g.md
# Measured: padding-cut, rl on a100-40g
| generate | 1.9460 | 1.9460 | +0.0% | 2.1700 | 2.1542 | -0.7% |
| logprobs | 0.1415 | 0.1695 | +19.8% | 0.1643 | 0.1678 | +2.2% |
| train | 0.2956 | 0.2598 | -12.1% | 0.3789 | 0.3319 | -12.4% |
| step | 2.3830 | 2.3753 | -0.3% | 2.7354 | 2.6760 | -2.2% |
- speedup: p

# M3S15's cell:

In [ ]:
%%bash
# m3s15: DecisionReport 1, the padding cut, on the T4 and the A100. `cost validate` saves each
# GPU's whole-step errors as validation.json beside its calibration.json: the reports' error
# band (it reads the runs' telemetry from Drive: a few minutes each). `cost compare` then
# predicts each experiment's step as the loop runs it and under the cut: M2's exit RL (× at
# bands 3 and 4) for m1-m3, warmstart.yaml (the 100M-token warm starts M4 needs) for m1-m4.
# Nothing runs on the GPU; reports go to $SCALELAB_HOME/reports/ on Drive, nothing to the
# registry. The greps keep each report's table rows and bullets.
set -f  # the * and [3,4] below are values, not files to match
cd /content/scalelab
for hardware in t4 a100-40g; do
  uv run scalelab cost validate --hardware $hardware | grep -E '^(step |verdict|saved)'
  for model in m1 m2 m3; do
    uv run scalelab cost compare configs/experiments/rl.yaml model=ladder_s/$model \
      task.ops=* task.difficulties=[3,4] --hardware $hardware | grep -E '^(# |\| [glst][a-z]+ |- )'
  done
  for model in m1 m2 m3 m4; do
    uv run scalelab cost compare configs/experiments/warmstart.yaml model=ladder_s/$model \
      --hardware $hardware | grep -E '^(# |\| [glst][a-z]+ |- )'
  done
done

step               28    4.0%   +0.6%  -10.1%
verdict:       step MAPE 4.0%, within the 15% M3's exit asks for
saved:         /content/drive/MyDrive/scalelab/bench/t4/validation.json
# DecisionReport: padding-cut, rl on t4
| generate | 1.4018 | 1.4018 | +0.0% | launch | gemv-decode |
| logprobs | 0.1790 | 0.1697 | -5.2% | memory | elementwise |
| train | 0.3323 | 0.2828 | -14.9% | memory | elementwise |
| step | 1.9131 | 1.8543 | -3.1% | | |
- speedup: 1.03x, 1,882 -> 1,941 steps an hour
- FLOPs a step: 8.7e+11 -> 7.31e+11 (-15.9%): padding is computed, and counted
- error band: whole steps here land within 4.0% on average (bias +0.6%, worst -10.1%, 28 runs): the candidate's step 1.7797-1.9289 s
- verdict: not worth building: the saving, 3.1%, is within the error
# DecisionReport: padding-cut, rl on t4
| generate | 2.5313 | 2.5313 | +0.0% | launch | gemv-decode |
| logprobs | 0.7125 | 0.6133 | -13.9% | compute | gemm |
| train | 1.1549 | 0.9602 | -16.9% | compute | gemm |
| step | 4.39

# M3S14's three cells:

> The following three code cells are from M3, Step 14: the cost model on an A100. The cells after them are older.

Cell G

In [ ]:
%%bash
# m3s14, first: the cost model of this A100, made as m3s12 made the T4's. `scalelab bench` times
# the ops of m1-m4 on this GPU, shaped as their layers run them; `scalelab cost calibrate` then
# fits the efficiencies to those timings. Both save under $SCALELAB_HOME/bench/a100-40g/ on Drive
# and record nothing in the registry (the T4 took 29 s to bench). After a disconnection, run it
# again from the top.
set -euo pipefail
cd /content/scalelab
uv run scalelab bench --hardware a100-40g
uv run scalelab cost calibrate --hardware a100-40g

hardware:      a100-40g
models:        ladder_s/m1 ladder_s/m2 ladder_s/m3 ladder_s/m4
results:       /content/drive/MyDrive/scalelab/bench/a100-40g
gemm:          107 cases: 107 timings in 2 s
copy:          8 cases: 8 timings in 0 s
elementwise:   48 cases: 84 timings in 4 s
attention:     112 cases: 160 timings in 2 s
decode:        64 cases: 64 timings in 18 s
hardware:      a100-40g
measured on:   torch 2.14.1, cuda 13.0, device NVIDIA A100-SXM4-40GB
samples:       423 from /content/drive/MyDrive/scalelab/bench/a100-40g
group           compute  memory     launch   (1.000: never the limit)
gemm              0.846   1.000   15.51 us
gemv-decode       0.988   0.277   95.02 us
attention         0.476   0.370   68.61 us
copy              1.000   0.972    6.70 us
norm              1.000   0.541   79.94 us
norm.grad         1.000   0.152  302.76 us
rope              1.000   0.347  126.96 us
rope.grad         1.000   0.662  157.68 us
act               1.000   0.558    7.96 us
act.grad    

Cell H

In [ ]:
%%bash
# m3s14: the held-out steps on the A100, chosen before the fit, which never sees a whole step:
# the T4's cell E again (m1-m4 warm starts of 1M tokens, which time their steps; 5 short RL runs
# from M1's exit warm starts, each changing one thing M2's exit kept fixed), then M2's exit
# settings for 20 steps (× at bands 3 and 4: GRPO for m1, m2 and m3, RLOO for m2), which the T4
# ran for 200. 13 runs; the RL ones graded at step 0 and at the end on 100 problems a band
# (gradings aren't timed). A guess: under an hour. After a disconnection, run the settings,
# Drive and install cells, then this one again: finished runs say "already done", and the
# interrupted one resumes from its last checkpoint.
set -f  # no filename patterns here: the * and [3,4] below are values, not files to match
cd /content/scalelab
LOCAL=runtime.local_registry=/content/registry.db
for model in m1 m2 m3 m4; do
  echo "=== warm start $model"
  uv run scalelab run configs/experiments/warmstart.yaml "model=ladder_s/$model" \
    warm_start.tokens=1000000 eval.every=0 eval.problems=100 $LOCAL
done
SHORT="rl.max_steps=20 eval.every=0 eval.problems=100 $LOCAL"
MUL="task.ops=* task.difficulties=[3,4]"
for run in "m4 1729c99162d2c661 $MUL" \
  "m2 71cd3add8c11e667 task.difficulties=[2]" \
  "m2 71cd3add8c11e667 task.ops=* task.difficulties=[5]" \
  "m1 c73d1d0f4e234964 $MUL rl.prompts_per_step=128" \
  "m3 52606c3253c9e849 $MUL algo=rloo rl.epochs=2" \
  "m1 c73d1d0f4e234964 $MUL" \
  "m2 71cd3add8c11e667 $MUL" \
  "m3 52606c3253c9e849 $MUL" \
  "m2 71cd3add8c11e667 $MUL algo=rloo"; do
  set -- $run  # $1 the model, $2 its warm start's config hash, then what this run changes
  model=$1 warm=$2
  shift 2
  echo "=== $model $*"
  uv run scalelab run configs/experiments/rl.yaml "model=ladder_s/$model" \
    "rl.from_run=$warm-s0-2aeaf16b" "$@" $SHORT
done

=== warm start m1
experiment:    warmstart (config 972a1dbc80ffb8fe), seed 0
run:           972a1dbc80ffb8fe-s0-218c21a6
code:          218c21a6 (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
metrics:       /content/drive/MyDrive/scalelab/metrics/972a1dbc80ffb8fe-s0-218c21a6
started
step 188: loss 0.950  arithmetic-2 0.0%  arithmetic-3 0.0%  arithmetic-4 0.0%  arithmetic-5 0.0%  (70,508 tokens/s, about 0 s to go)
done: ran 188 steps in 15 s (1,001,779 tokens and 1.55e+13 FLOPs in all), last loss 0.9499361515045166
=== warm start m2
experiment:    warmstart (config 9c3b171457d8f80a), seed 0
run:           9c3b171457d8f80a-s0-218c21a6
code:          218c21a6 (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
metrics:       /content/drive/MyDrive/scalelab/metrics/9c3b171457d8f80a-s0-218c21a6
started


Cell I:

In [ ]:
%%bash
# m3s14: the verdict on this A100. The cost model's step, phase by phase, against every run on an
# A100 that timed its steps: cell H's 13, which the fit never saw. Declared before the runs: the
# second GPU's half of M3's exit is met if the step MAPE is at most 15%. Reading the runs'
# telemetry from Drive takes a few minutes.
cd /content/scalelab
uv run scalelab cost validate --hardware a100-40g

hardware:      a100-40g
calibration:   /content/drive/MyDrive/scalelab/bench/a100-40g/calibration.json
runs:          13 runs timed their steps
run                            steps  generate  logprobs     train      step  step, predicted vs measured
972a1dbc80ffb8fe-s0-218c21a6     187                         -0.6%    -41.5%  0.016 s vs 0.027 s
9c3b171457d8f80a-s0-218c21a6     187                         -8.1%    -24.3%  0.049 s vs 0.064 s
a39fba4a58c4648c-s0-218c21a6     187                         +1.5%     -7.9%  0.113 s vs 0.123 s
0466fdbdfa804a0c-s0-218c21a6     187                         +0.5%     -4.5%  0.225 s vs 0.236 s
513bd948ab95f41e-s0-218c21a6      19     +3.3%     +4.9%     -3.2%     +1.4%  5.655 s vs 5.576 s
7f9eeec6c814c6b3-s0-218c21a6      19    -14.2%     +0.5%    -30.6%    -19.9%  0.590 s vs 0.737 s
c407d55fd43af2bb-s0-218c21a6      19     +0.9%     -4.0%    -19.5%     -4.2%  2.609 s vs 2.723 s
c094b96b92030c3c-s0-218c21a6      19     -6.6%    -27.5%    -34.2%    -

# M3S12's three cells:

> The following three code cells are from M3, Step 12: the cost model on the T4. The cells after them are older.

Cell D

In [ ]:
%%bash
# m3s12, first: the cost model of this T4. `scalelab bench` times the ops of m1-m4 on this GPU,
# shaped as their layers run them (matmuls, copies, norms, attention, whole decode steps);
# `scalelab cost calibrate` then fits the efficiencies to those timings. Both save under
# $SCALELAB_HOME/bench/t4/ on Drive and record nothing in the registry. The bench has never run
# on a GPU before: a guess, under 10 minutes. After a disconnection, run it again from the top.
set -euo pipefail
cd /content/scalelab
uv run scalelab bench --hardware t4
uv run scalelab cost calibrate --hardware t4

hardware:      t4
models:        ladder_s/m1 ladder_s/m2 ladder_s/m3 ladder_s/m4
results:       /content/drive/MyDrive/scalelab/bench/t4
gemm:          107 cases: 107 timings in 2 s
copy:          8 cases: 8 timings in 0 s
elementwise:   48 cases: 84 timings in 4 s
attention:     112 cases: 160 timings in 3 s
decode:        64 cases: 64 timings in 20 s
hardware:      t4
measured on:   torch 2.14.1, cuda 13.0, device Tesla T4
samples:       423 from /content/drive/MyDrive/scalelab/bench/t4
group           compute  memory     launch   (1.000: never the limit)
gemm              0.508   0.348   19.02 us
gemv-decode       1.000   0.927  113.88 us
attention         0.127   0.300   60.52 us
copy              1.000   0.757    6.61 us
norm              1.000   0.230   50.99 us
norm.grad         1.000   0.085  337.34 us
rope              1.000   0.155   66.20 us
rope.grad         1.000   0.293  274.32 us
act               1.000   0.437    0.39 us
act.grad          1.000   0.430  129.30 us
residu

Cell E

In [ ]:
%%bash
# m3s12: the held-out steps, chosen before the fit, which never sees a whole step. Warm starts
# time their steps since m3s11 (M1's exit warm starts timed nothing): m1-m4 again, 1M tokens
# each, otherwise as M1's exit set them up. Then 5 short RL runs from M1's exit warm starts, each
# changing something M2's exit (m1-m3, 64 x 8, × at bands 3 and 4) kept fixed: the size (m4),
# short answers (every operation at band 2), long answers (× at band 5), twice the rollouts
# (128 x 8), two updates a step (RLOO, epochs 2). 20 steps each, graded at step 0 and at the
# end on 100 problems a band (gradings aren't timed). About half an hour. After a disconnection,
# run the settings, Drive and install cells, then this one again: finished runs say "already
# done", and the interrupted one resumes from its last checkpoint.
set -f  # no filename patterns here: the * and [3,4] below are values, not files to match
cd /content/scalelab
LOCAL=runtime.local_registry=/content/registry.db
for model in m1 m2 m3 m4; do
  echo "=== warm start $model"
  uv run scalelab run configs/experiments/warmstart.yaml "model=ladder_s/$model" \
    warm_start.tokens=1000000 eval.every=0 eval.problems=100 $LOCAL
done
SHORT="rl.max_steps=20 eval.every=0 eval.problems=100 $LOCAL"
for run in "m4 1729c99162d2c661 task.ops=* task.difficulties=[3,4]" \
  "m2 71cd3add8c11e667 task.difficulties=[2]" \
  "m2 71cd3add8c11e667 task.ops=* task.difficulties=[5]" \
  "m1 c73d1d0f4e234964 task.ops=* task.difficulties=[3,4] rl.prompts_per_step=128" \
  "m3 52606c3253c9e849 task.ops=* task.difficulties=[3,4] algo=rloo rl.epochs=2"; do
  set -- $run  # $1 the model, $2 its warm start's config hash, then what this run changes
  model=$1 warm=$2
  shift 2
  echo "=== $model $*"
  uv run scalelab run configs/experiments/rl.yaml "model=ladder_s/$model" \
    "rl.from_run=$warm-s0-2aeaf16b" "$@" $SHORT
done

=== warm start m1
experiment:    warmstart (config 972a1dbc80ffb8fe), seed 0
run:           972a1dbc80ffb8fe-s0-5f58a897
code:          5f58a897 (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
metrics:       /content/drive/MyDrive/scalelab/metrics/972a1dbc80ffb8fe-s0-5f58a897
started
step 188: loss 0.954  arithmetic-2 0.0%  arithmetic-3 0.0%  arithmetic-4 0.0%  arithmetic-5 0.0%  (46,564 tokens/s, about 0 s to go)
done: ran 188 steps in 22 s (1,001,779 tokens and 1.55e+13 FLOPs in all), last loss 0.9535229206085205
=== warm start m2
experiment:    warmstart (config 9c3b171457d8f80a), seed 0
run:           9c3b171457d8f80a-s0-5f58a897
code:          5f58a897 (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
metrics:       /content/drive/MyDrive/scalelab/metrics/9c3b171457d8f80a-s0-5f58a897
started


Cell F

In [ ]:
%%bash
# m3s12: the verdict on this T4. The cost model's step, phase by phase, against every run on a
# T4 that timed its steps: M2's exit runs (RL has timed its steps since m2s29) and cell E's. The
# fit saw none of them. Declared before the runs: the T4 half of M3's exit is met if the step
# MAPE is at most 15%. Reading every run's telemetry from Drive takes a few minutes.
cd /content/scalelab
uv run scalelab cost validate --hardware t4

hardware:      t4
calibration:   /content/drive/MyDrive/scalelab/bench/t4/calibration.json
runs:          28 runs timed their steps (left out 15: no timed steps, or a config that no longer loads)
run                            steps  generate  logprobs     train      step  step, predicted vs measured
f00297f7c071356a-s0-4da66cf3     199    +20.0%     -2.6%    -17.8%     -0.0%  8.567 s vs 8.570 s
f0212b5cc4136e6b-s0-4da66cf3     199     -3.8%     +5.9%    -13.5%     -5.8%  1.913 s vs 2.031 s
db53dee96d95c1df-s0-4da66cf3     241     -4.7%     +6.5%    -10.7%     -6.4%  1.824 s vs 1.949 s
2c299e1c7b08dc26-s0-4da66cf3     199    +17.0%     +1.1%     -5.6%     +7.0%  4.399 s vs 4.113 s
7ffdd9d8beb609f8-s0-4da66cf3     242    +18.5%     +1.8%    -11.9%     +5.9%  4.042 s vs 3.818 s
f0212b5cc4136e6b-s1-4da66cf3     199     -2.7%     +4.7%     -3.9%     -3.3%  1.913 s vs 1.979 s
db53dee96d95c1df-s1-4da66cf3     241     -3.7%     +7.4%     -1.7%     -3.9%  1.824 s vs 1.898 s
2c299e1c7b08dc26-s1

# M2S40's three cells:

> The following three code cells are from M2, Step 40: the M2 exit. The cells after them are older.

Cell A

In [ ]:
%%bash
# m2s40, first: m3's FLOPs budget for the exit, the FLOPs of its first 200 GRPO steps (the
# m2s25 probe measured m1's and m2's: 1.71e14 and 9.68e14). The probe's settings at m3: GRPO,
# lr 1e-5, × at bands 3 and 4, 200 steps, graded every 50. About 35 minutes. Its last line,
# "done: ran 200 steps in … (N FLOPs in all)", gives N: m3's budget in the next cell.
set -f  # no filename patterns here: the * and [3,4] below are values, not files to match
cd /content/scalelab
uv run scalelab run configs/experiments/rl.yaml model=ladder_s/m3 \
  rl.from_run=52606c3253c9e849-s0-2aeaf16b task.ops=* task.difficulties=[3,4] \
  rl.max_steps=200 rl.flops=1e16 eval.every=50 runtime.local_registry=/content/registry.db

experiment:    rl (config f00297f7c071356a), seed 0
run:           f00297f7c071356a-s0-4da66cf3
code:          4da66cf3 (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
warm start:    52606c3253c9e849-s0-2aeaf16b
metrics:       /content/drive/MyDrive/scalelab/metrics/f00297f7c071356a-s0-4da66cf3
gradings:      /content/drive/MyDrive/scalelab/gradings/f00297f7c071356a-s0-4da66cf3
started
step 0 (the warm start): arithmetic-3 90.4% (sampled 85.2%)  arithmetic-4 76.8% (sampled 73.4%)
step 50: reward 0.484  arithmetic-3 92.7% (sampled 87.4%)  arithmetic-4 80.1% (sampled 75.0%)  (1.48e+12 FLOPs/s, about 24 min to go)
step 100: reward 0.383  arithmetic-3 93.2% (sampled 88.1%)  arithmetic-4 80.9% (sampled 75.7%)  (1.47e+12 FLOPs/s, about 16 min to go)
step 105: WARN spike in optim/grad_norm: value 0.1469  median 0.06742  mad 0.008728  z 6.144
step 117: WARN spike in optim/loss: value 0.0003523  med

Cell B

In [ ]:
%%bash
# m2s40: M2's exit. GRPO and RLOO from M1's exit warm starts of m1, m2 and m3, 3 seeds each:
# 18 runs, about 6½ hours on a T4. Every run of a size gets the same FLOPs budget, that of
# 200 GRPO steps (RLOO, with no KL reference to score, gets a few more steps). × at bands 3
# and 4, graded every 50 steps. Seed 0 of all six goes first, then seed 1, then seed 2.
# After a disconnection, run the settings, Drive and install cells, then this one again:
# finished runs say "already done", and the interrupted one resumes from its last checkpoint.
set -f  # no filename patterns here: the * and [3,4] below are values, not files to match
M3_FLOPS="2.79e+15"  # m3's budget: the FLOPs the cell above printed, to 3 digits (like 1.71e14)
: "${M3_FLOPS:?write the budget of m3 into M3_FLOPS first}"
cd /content/scalelab
EXIT="task.ops=* task.difficulties=[3,4] eval.every=50 runtime.local_registry=/content/registry.db"
for seed in 0 1 2; do
  for size in "m1 c73d1d0f4e234964 1.71e14" "m2 71cd3add8c11e667 9.68e14" \
    "m3 52606c3253c9e849 $M3_FLOPS"; do
    set -- $size  # $1 the model, $2 its warm start's config hash, $3 its FLOPs budget
    for algo in grpo rloo; do
      echo "=== $1 $algo, seed $seed"
      uv run scalelab run configs/experiments/rl.yaml "model=ladder_s/$1" "algo=$algo" \
        "seed=$seed" "rl.from_run=$2-s0-2aeaf16b" "rl.flops=$3" $EXIT
    done
  done
done

=== m1 grpo, seed 0
experiment:    rl (config f0212b5cc4136e6b), seed 0
run:           f0212b5cc4136e6b-s0-4da66cf3
code:          4da66cf3 (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
already done: nothing to do
=== m1 rloo, seed 0
experiment:    rl (config db53dee96d95c1df), seed 0
run:           db53dee96d95c1df-s0-4da66cf3
code:          4da66cf3 (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
already done: nothing to do
=== m2 grpo, seed 0
experiment:    rl (config 2c299e1c7b08dc26), seed 0
run:           2c299e1c7b08dc26-s0-4da66cf3
code:          4da66cf3 (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
already done: nothing to do
=== m2 rloo, seed 0
experiment:    rl (config 7ffdd9d8beb609f

Cell C

In [ ]:
%%bash
# m2s40: the exit's verdict. For each experiment, what RL did to greedy pass@1 between step 0
# (the warm start) and the last grading, its seeds paired problem by problem, with 95%
# bootstrap intervals. The exit's lines: arithmetic-3/* and arithmetic-4/* of each model and
# algorithm (the + and - lines: did training on × cost the other operations?).
cd /content/scalelab
uv run scalelab report gains

m3 grpo: experiment f00297f7c071356a at 4da66cf3
  seeds 0; last steps 200
  eval set            n   first    last  gain, mean [95% CI]       gain, IQM [95% CI]        verdict
  arithmetic-3     1000   90.4%   93.9%  +3.5 [+2.1, +5.0] pp      +3.5 [+2.1, +5.0] pp      improved
  arithmetic-3/*    329   70.8%   81.5%  +10.6 [+6.4, +14.9] pp    +10.6 [+6.4, +14.9] pp    improved
  arithmetic-3/+    352  100.0%  100.0%  +0.0 [+0.0, +0.0] pp      +0.0 [+0.0, +0.0] pp      unclear
  arithmetic-3/-    319  100.0%  100.0%  +0.0 [+0.0, +0.0] pp      +0.0 [+0.0, +0.0] pp      unclear
  arithmetic-4     1000   76.8%   81.4%  +4.6 [+3.0, +6.3] pp      +4.6 [+3.0, +6.3] pp      improved
  arithmetic-4/*    316   27.2%   41.8%  +14.6 [+9.5, +19.6] pp    +14.6 [+9.5, +19.6] pp    improved
  arithmetic-4/+    342  100.0%  100.0%  +0.0 [+0.0, +0.0] pp      +0.0 [+0.0, +0.0] pp      unclear
  arithmetic-4/-    342   99.4%   99.4%  +0.0 [+0.0, +0.0] pp      +0.0 [+0.0, +0.0] pp      unclear

m1 grpo: ex

# *Older_2*

> The following code cell is from M2, Step 36. The cells after it are older. date : 23oct26

In [ ]:
%%bash
# m2s36: what telemetry costs at m3 on a T4 (the budget: 5% of step time). From M1's m3 warm
# start, × at bands 3 and 4: 5 steps to warm up, then 30 timed, first without telemetry, then
# with it. About 15 minutes. Records nothing; exits with an error if telemetry is over budget.
set -f  # no filename patterns here: the * and [3,4] below are values, not files to match
cd /content/scalelab
uv run scalelab overhead configs/experiments/rl.yaml model=ladder_s/m3 \
  rl.from_run=52606c3253c9e849-s0-2aeaf16b task.ops=* task.difficulties=[3,4]

experiment:    rl (config 3ea839dc34ca6969), seed 0
warm start:    52606c3253c9e849-s0-2aeaf16b
device:        Tesla T4
steps:         5 to warm up, then 30 timed; without, then with telemetry
without:       8.715 s a step
with:          8.936 s a step (0 detector events)
overhead:      +2.5% of step time, within the 5% budget


# *Older_1*

> *The following two code cells are from M2, Step 25. The cells after it are older. 21oct26/00:45 -> 01:50*

In [ ]:
%%bash
# M2's first RL probe (m2s25): GRPO from M1's exit warm starts, trained on multiplication (×) at
# bands 3 and 4: 200 steps of 64 problems x 8 answers, graded every 50 steps. Five runs, about
# an hour on a T4. After a disconnection, run this cell again: finished runs say "already
# done", and the interrupted one resumes from its last checkpoint.
set -f  # no filename patterns here: the * and [3,4] below are values, not files to match
cd /content/scalelab
PROBE="task.ops=* task.difficulties=[3,4] rl.max_steps=200 rl.flops=1e16 eval.every=50"
for run in "m1 c73d1d0f4e234964 1e-5" "m1 c73d1d0f4e234964 3e-5" "m1 c73d1d0f4e234964 1e-4" \
  "m2 71cd3add8c11e667 1e-5" "m2 71cd3add8c11e667 3e-5"; do
  set -- $run  # $1 the model, $2 its warm start's config hash, $3 the learning rate
  echo "=== $1, lr $3"
  uv run scalelab run configs/experiments/rl.yaml "model=ladder_s/$1" \
    "rl.from_run=$2-s0-2aeaf16b" "rl.lr=$3" $PROBE runtime.local_registry=/content/registry.db
done

=== m1, lr 1e-5
experiment:    rl (config bc6d13d7603738d2), seed 0
run:           bc6d13d7603738d2-s0-c851c9c6
code:          c851c9c6 (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
warm start:    c73d1d0f4e234964-s0-2aeaf16b
started
step 0 (the warm start): arithmetic-3 80.1% (sampled 72.1%)  arithmetic-4 69.9% (sampled 67.1%)
step 50: reward 0.131  arithmetic-3 81.3% (sampled 73.0%)  arithmetic-4 71.3% (sampled 67.2%)  (3.43e+11 FLOPs/s, about 6 min to go)
step 100: reward 0.086  arithmetic-3 83.3% (sampled 73.5%)  arithmetic-4 70.8% (sampled 67.3%)  (3.43e+11 FLOPs/s, about 4 min to go)
step 150: reward 0.127  arithmetic-3 83.5% (sampled 74.0%)  arithmetic-4 70.3% (sampled 67.0%)  (3.41e+11 FLOPs/s, about 2 min to go)
step 200: reward 0.184  arithmetic-3 83.3% (sampled 73.9%)  arithmetic-4 70.0% (sampled 67.7%)  (3.39e+11 FLOPs/s, about 0 s to go)
done: ran 200 steps in 9 min (1.71e+14

In [ ]:
%%bash
# The probe's grades at bands 3 and 4, step by step: multiplication greedy, sampled pass@1 and
# pass@4, then addition and subtraction greedy (RL trains on × alone: + and - must hold).
cd /content/scalelab
uv run python - <<'PYTHON'
import json

from scalelab.core.registry import Registry

registry = Registry.sqlite("/content/registry.db")
for run in registry.list_runs():
    config = json.loads(registry.experiment_json(run.config_hash) or "{}")
    rl = config.get("rl") or {}
    if rl.get("max_steps") != 200 or config["task"].get("ops") != "*":
        continue  # not one of the probe's runs
    took = ""
    if run.ended_at is not None:
        took = f", {(run.ended_at - run.started_at).total_seconds() / 60:.0f} min"
    print(f"{config['model']['name']}  lr {rl['lr']:g}  {run.status}{took}  ({run.run_id})")
    evals = registry.list_evals(run.run_id)
    for step in sorted({e.step for e in evals}):
        value = {(e.task, e.metric): e.value for e in evals if e.step == step}
        for band in (3, 4):
            name = f"arithmetic-{band}"
            times = (
                f"× {value[name + '/*', 'greedy_pass@1']:6.1%}"
                f"  sampled {value[name + '/*', 'sampled_pass@1']:6.1%}"
                f"  @4 {value[name + '/*', 'sampled_pass@4']:6.1%}"
            )
            others = (
                f"+ {value[name + '/+', 'greedy_pass@1']:6.1%}"
                f"  - {value[name + '/-', 'greedy_pass@1']:6.1%}"
            )
            print(f"  step {step:>3}  band {band}:  {times}    {others}")
PYTHON

m1  lr 1e-05  done, 9 min  (bc6d13d7603738d2-s0-c851c9c6)
  step   0  band 3:  ×  41.0%  sampled  18.4%  @4  45.9%    + 100.0%  -  98.4%
  step   0  band 4:  ×   8.5%  sampled   2.0%  @4   7.3%    + 100.0%  -  96.5%
  step  50  band 3:  ×  44.7%  sampled  20.7%  @4  47.1%    + 100.0%  -  98.4%
  step  50  band 4:  ×  13.3%  sampled   2.7%  @4   9.5%    + 100.0%  -  96.2%
  step 100  band 3:  ×  50.8%  sampled  22.9%  @4  50.2%    + 100.0%  -  98.4%
  step 100  band 4:  ×  12.0%  sampled   2.8%  @4  10.1%    + 100.0%  -  95.9%
  step 150  band 3:  ×  51.4%  sampled  23.9%  @4  51.7%    + 100.0%  -  98.4%
  step 150  band 4:  ×  10.4%  sampled   2.2%  @4   7.9%    + 100.0%  -  95.9%
  step 200  band 3:  ×  50.8%  sampled  23.9%  @4  50.2%    + 100.0%  -  98.4%
  step 200  band 4:  ×   9.5%  sampled   3.0%  @4  10.1%    + 100.0%  -  95.9%
m1  lr 3e-05  done, 9 min  (db791584d9496335-s0-c851c9c6)
  step   0  band 3:  ×  41.0%  sampled  18.4%  @4  45.9%    + 100.0%  -  98.4%
  step   0  ban

# *Older*

In [ ]:
%%bash
# Run it. The live registry stays on the VM's own disk (SQLite over the Drive mount is
# unreliable), and scalelab copies it to Drive after every change.
cd /content/scalelab
uv run scalelab run "$EXPERIMENT" $OVERRIDES runtime.local_registry=/content/registry.db

experiment:    hello (config c6b6e17b3e7b116b), seed 0
run:           c6b6e17b3e7b116b-s0-2aeaf16b
code:          2aeaf16b (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
resuming (it was running)
done: the last checkpoint was already saved, so there was nothing left to train


In [ ]:
%%bash
# Every run in the registry, with its grades.
cd /content/scalelab
uv run python - <<'EOF'
from scalelab.core.registry import Registry

registry = Registry.sqlite("/content/registry.db")
for run in registry.list_runs():
    print(run.run_id, run.status, run.hardware)
    for e in registry.list_evals(run.run_id):
        print(f"  step {e.step:>6}  {e.task}  {e.value:.1%} ± {e.stderr:.1%}  (n={e.n})")
EOF

c6b6e17b3e7b116b-s0-81795f9e done Tesla T4
  step     16  arithmetic-2  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-3  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-4  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-5  0.0% ± 0.0%  (n=20)
c6b6e17b3e7b116b-s0-2aeaf16b done Tesla T4
  step     16  arithmetic-2  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-3  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-4  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-5  0.0% ± 0.0%  (n=20)
47219a7a4a207d7d-s0-2aeaf16b done Tesla T4
  step    100  arithmetic-2  1.0% ± 1.0%  (n=100)
  step    100  arithmetic-3  0.0% ± 0.0%  (n=100)
  step    100  arithmetic-4  0.0% ± 0.0%  (n=100)
  step    100  arithmetic-5  0.0% ± 0.0%  (n=100)
  step    200  arithmetic-2  4.0% ± 2.0%  (n=100)
  step    200  arithmetic-3  1.0% ± 1.0%  (n=100)
  step    200  arithmetic-4  0.0% ± 0.0%  (n=100)
  step    200  arithmetic-5  0.0% ± 0.0%  (n=100)
  step    300  arithmetic-2  4.0% ± 2.0%  (n=100)
  step    300  arithmetic-3  

In [ ]:
%%bash
# M1's exit: warm-start m1, m2, m3 and m4 one after another (about 2½ hours on a T4).
# After a disconnection, run it again: finished runs say "already done", and the
# interrupted one resumes from its last checkpoint.
cd /content/scalelab
for model in m1 m2 m3 m4; do
  uv run scalelab run configs/experiments/warmstart.yaml "model=ladder_s/$model" \
    warm_start.tokens=20000000 runtime.local_registry=/content/registry.db
done

experiment:    warmstart (config c73d1d0f4e234964), seed 0
run:           c73d1d0f4e234964-s0-2aeaf16b
code:          2aeaf16b (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
started
step 1,000: loss 0.092  arithmetic-2 64.3%  arithmetic-3 61.9%  arithmetic-4 59.9%  arithmetic-5 54.7%  (54,982 tokens/s, about 4 min to go)
step 2,000: loss 0.057  arithmetic-2 77.9%  arithmetic-3 64.9%  arithmetic-4 63.0%  arithmetic-5 56.4%  (58,676 tokens/s, about 3 min to go)
step 3,000: loss 0.038  arithmetic-2 88.9%  arithmetic-3 72.3%  arithmetic-4 67.3%  arithmetic-5 64.3%  (58,320 tokens/s, about 71 s to go)
step 3,790: loss 0.025  arithmetic-2 94.5%  arithmetic-3 80.1%  arithmetic-4 69.9%  arithmetic-5 65.3%  (53,824 tokens/s, about 0 s to go)
done: ran 3790 steps in 6 min (20,003,027 tokens and 3.12e+14 FLOPs in all), last loss 0.024546917527914047
experiment:    warmstart (config 71cd3add8c11e667),

In [ ]:
%%bash
# The final grades of every warm start, oldest first.
cd /content/scalelab
uv run python - <<'EOF'
import json

from scalelab.core.registry import Registry

registry = Registry.sqlite("/content/registry.db")
for run in registry.list_runs():
    config = json.loads(registry.experiment_json(run.config_hash) or "{}")
    evals = registry.list_evals(run.run_id)
    if config.get("name") != "warmstart" or run.status != "done" or not evals:
        continue
    last = max(e.step for e in evals)
    grades = "  ".join(f"{e.task} {e.value:.1%}" for e in evals if e.step == last)
    tokens = config["warm_start"]["tokens"]
    print(f"{config['model']['name']}  {tokens:>11,} tokens  step {last:>5}:  {grades}")
EOF

m1    2,000,000 tokens  step   379:  arithmetic-2 8.0%  arithmetic-3 4.0%  arithmetic-4 2.0%  arithmetic-5 0.0%
m2    2,000,000 tokens  step   379:  arithmetic-2 56.0%  arithmetic-3 51.0%  arithmetic-4 41.0%  arithmetic-5 30.0%
m3    2,000,000 tokens  step   379:  arithmetic-2 44.0%  arithmetic-3 18.0%  arithmetic-4 2.0%  arithmetic-5 1.0%
m4    2,000,000 tokens  step   379:  arithmetic-2 12.0%  arithmetic-3 1.0%  arithmetic-4 0.0%  arithmetic-5 0.0%
m1   20,000,000 tokens  step  3790:  arithmetic-2 94.5%  arithmetic-3 80.1%  arithmetic-4 69.9%  arithmetic-5 65.3%
m2   20,000,000 tokens  step  3790:  arithmetic-2 98.3%  arithmetic-3 85.6%  arithmetic-4 76.2%  arithmetic-5 66.9%
m3   20,000,000 tokens  step  3790:  arithmetic-2 98.7%  arithmetic-3 90.4%  arithmetic-4 76.8%  arithmetic-5 66.4%
m4   20,000,000 tokens  step  3790:  arithmetic-2 98.3%  arithmetic-3 88.5%  arithmetic-4 77.4%  arithmetic-5 67.8%
